In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv


In [2]:
# Cell 0 
!pip install -q unsloth transformers peft accelerate datasets bitsandbytes trl
!pip install -q evaluate rouge_score sacrebleu sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Cell 1 
import subprocess
subprocess.run(["pip", "install", "-q", "evaluate", "rouge_score", "sacrebleu"], check=True)

import os
import torch
import math
import json
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
from abc import ABC, abstractmethod
import evaluate

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("All imports OK")
print(f"GPUs: {torch.cuda.device_count()}")

2026-03-05 05:56:28.139529: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772690188.509070      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772690188.615733      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772690189.545475      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772690189.545513      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772690189.545515      25 computation_placer.cc:177] computation placer alr

All imports OK
GPUs: 2


In [4]:
df = pd.read_csv("/kaggle/input/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv")


In [5]:
df=df.head(5000)

In [6]:
!pip install -q unsloth transformers peft accelerate datasets bitsandbytes evaluate rouge_score sacrebleu sentencepiece


In [ ]:
# Cell 3 — HuggingFace login
from huggingface_hub import login

HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX" 
login(token=HF_TOKEN)

In [8]:
class DatasetProcessor:
    def __init__(self, model_name, max_seq_length=4096):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
        self.max_seq_length = max_seq_length
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def preprocess(self, df):
        df = df.dropna().head(5000).reset_index(drop=True)
    
        def make_text(row):
            messages = [
                {
                    "role": "system",
                    "content": "আপনি একজন সহানুভূতিশীল বাংলা সহায়তাকারী। সবসময় বাংলায় উত্তর দিন।"
                },
                {
                    "role": "user",
                    "content": f"বিষয়: {row['Topics']}\n\nশিরোনাম: {row['Question-Title']}\n\nপ্রশ্ন: {row['Questions']}"
                },
                {
                    "role": "assistant",
                    "content": row['Answers']
                }
            ]
            return self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
    
        df["text"] = df.apply(make_text, axis=1)
        df = df.dropna(subset=["text"]).reset_index(drop=True)  # ← add this line
    
        def truncate(text):
            tokens = self.tokenizer(
                text,
                truncation=True,
                max_length=self.max_seq_length,
                return_tensors=None
            )
            return self.tokenizer.decode(tokens["input_ids"], skip_special_tokens=False)
    
        df["text"] = df["text"].apply(truncate)
        dataset = Dataset.from_pandas(df[["text"]])
        return dataset

In [9]:
# Cell 5 — split_dataset
from datasets import DatasetDict

def split_dataset(dataset, test_size=0.1, seed=42):
    split = dataset.train_test_split(test_size=test_size, seed=seed)
    return DatasetDict({
        "train": split["train"].shuffle(seed=seed),
        "eval":  split["test"].shuffle(seed=seed)
    })

In [10]:
# Cell 6 — Strategy ABC
from abc import ABC, abstractmethod

class FineTuningStrategy(ABC):
    @abstractmethod
    def train(self, dataset_dict):
        pass

In [11]:
# Cell 7 — UnslothStrategy: Single GPU, maximum memory savings
import os
import torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

class UnslothStrategy(FineTuningStrategy):
    def __init__(self, model_name, max_seq_length=4096):  # ← REDUCED from 4096
        self.model_name     = model_name
        self.max_seq_length = max_seq_length
        self.model          = None
        self.tokenizer      = None
        self.train_loss     = None
        self.val_loss       = None

    def train(self, dataset_dict):
        # 1. Load model — Unsloth must stay on single GPU
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.model_name,
            max_seq_length=self.max_seq_length,
            load_in_4bit=True,
            token=HF_TOKEN,
            rope_scaling=None,
        )

        # 2. Apply LoRA — reduced rank to save memory
        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=8,                    # ← reduced from 16
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"],
            lora_alpha=16,          # ← reduced from 32
            lora_dropout=0.05,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=42,
        )

        # 3. Trainer — bare minimum memory footprint
        trainer = SFTTrainer(
            model=self.model,
            tokenizer=self.tokenizer,
            train_dataset=dataset_dict["train"],
            args=SFTConfig(
                dataset_text_field="text",
                max_seq_length=self.max_seq_length,

                per_device_train_batch_size=1,
                gradient_accumulation_steps=16,
                num_train_epochs=2,
                warmup_steps=5,
                logging_steps=10,

                eval_strategy="no",
                save_strategy="steps",

                output_dir="outputs",
                optim="adamw_8bit",
                fp16=True,
                max_grad_norm=0.3,
                prediction_loss_only=True,
                dataloader_num_workers=0,
                dataloader_pin_memory=False,
                report_to="none",
                packing=False,
    
            ),
        )

        torch.cuda.empty_cache()
        train_result    = trainer.train()
        self.train_loss = train_result.training_loss

        self.val_loss = self._compute_eval_loss(dataset_dict["eval"])
        return self.model, self.tokenizer

    def _compute_eval_loss(self, eval_dataset, n_samples=30):  # ← reduced samples
        import math
        self.model.eval()
        losses = []

        for example in eval_dataset.select(range(min(n_samples, len(eval_dataset)))):
            inputs = self.tokenizer(
                example["text"],
                return_tensors="pt",
                truncation=True,
                max_length=self.max_seq_length,
            ).to("cuda")

            with torch.no_grad():
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                losses.append(outputs.loss.item())

            del inputs, outputs
            torch.cuda.empty_cache()

        self.val_loss = sum(losses) / len(losses) if losses else None
        print(f"Eval Loss: {self.val_loss:.4f}")
        return self.val_loss

/tmp/ipykernel_25/1706676026.py:7: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [12]:
# Cell 8 — LLAMAFineTuner
class LLAMAFineTuner:
    def __init__(self, strategy: FineTuningStrategy):
        self.strategy = strategy

    def run(self, dataset_dict):
        return self.strategy.train(dataset_dict)

In [13]:
# Cell 9 — Evaluator (FIXED)
import evaluate
import torch
import math

class Evaluator:
    def __init__(self, model, tokenizer, max_new_tokens=256):
        self.model          = model
        self.tokenizer      = tokenizer
        self.max_new_tokens = max_new_tokens
        self.bleu  = evaluate.load("sacrebleu")
        self.rouge = evaluate.load("rouge")

    def generate_response(self, prompt: str) -> str:
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=3800
        ).to("cuda")

        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,         
                no_repeat_ngram_size=4,          
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
            )

        new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    def compute_perplexity(self, dataset, n_samples=20):
        self.model.eval()
        losses = []
        for example in dataset.select(range(min(n_samples, len(dataset)))):
            inputs = self.tokenizer(
                example["text"],
                return_tensors="pt",
                truncation=True,
                max_length=4096
            ).to("cuda")
            with torch.no_grad():
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                losses.append(outputs.loss.item())
        return math.exp(sum(losses) / len(losses))

    def evaluate_on_samples(self, dataset, n_samples=50):
        predictions, references = [], []

        ASSISTANT_SPLIT = "<|start_header_id|>assistant<|end_header_id|>"

        for i, example in enumerate(dataset.select(range(min(n_samples, len(dataset))))):
            text = example["text"]

            if i == 0:
                print("=== FIRST SAMPLE (first 400 chars) ===")
                print(repr(text[:400]))
                print("======================================")

            if ASSISTANT_SPLIT not in text:
                print(f"Sample {i}: split token not found, skipping")
                continue

            prompt_part, ref_part = text.split(ASSISTANT_SPLIT, 1)
            prompt = prompt_part + ASSISTANT_SPLIT

            reference = ref_part.replace("<|eot_id|>", "").strip()

            if not reference:
                print(f"Sample {i}: empty reference, skipping")
                continue

            pred = self.generate_response(prompt)
            predictions.append(pred)
            references.append(reference)

        print(f"\nEvaluated {len(predictions)} samples")
        if predictions:
            print("--- Example prediction ---")
            print(predictions[0][:200])
            print("--- Example reference ---")
            print(references[0][:200])

        return self.compute_metrics(predictions, references), predictions, references

    def compute_metrics(self, predictions, references):
        bleu = self.bleu.compute(
            predictions=predictions,
            references=[[r] for r in references]
        )
    
        rouge = self.rouge.compute(
            predictions=predictions,
            references=references,
            tokenizer=lambda x: list(x) 
        )
    
        return {
            "BLEU":   round(bleu["score"], 4),
            "ROUGE1": round(rouge["rouge1"], 4),
            "ROUGE2": round(rouge["rouge2"], 4),
            "ROUGEL": round(rouge["rougeL"], 4),
        }

In [14]:
# Cell 10 — ExperimentLogger  (FIXED: added log methods)
import sqlite3
import json
from datetime import datetime

class ExperimentLogger:
    def __init__(self, db_path="experiments.db"):
        self.conn = sqlite3.connect(db_path)
        self.create_tables()

    def create_tables(self):
        cur = self.conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS LLAMAExperiments (
                id          INTEGER PRIMARY KEY AUTOINCREMENT,
                model_name  TEXT,
                lora_config TEXT,
                train_loss  REAL,
                val_loss    REAL,
                metrics     TEXT,
                timestamp   TEXT
            )
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS GeneratedResponses (
                id              INTEGER PRIMARY KEY AUTOINCREMENT,
                experiment_id   INTEGER,
                input_text      TEXT,
                response_text   TEXT,
                timestamp       TEXT
            )
        """)
        self.conn.commit()

    def log_experiment(self, model_name, lora_config, train_loss,
                       val_loss, metrics) -> int:
        cur = self.conn.cursor()
        cur.execute("""
            INSERT INTO LLAMAExperiments
            (model_name, lora_config, train_loss, val_loss, metrics, timestamp)
            VALUES (?, ?, ?, ?, ?, ?)
        """, (
            model_name,
            json.dumps(lora_config),
            train_loss,
            val_loss,
            json.dumps(metrics),
            datetime.now().isoformat()
        ))
        self.conn.commit()
        return cur.lastrowid

    def log_response(self, experiment_id, input_text, response_text):
        cur = self.conn.cursor()
        cur.execute("""
            INSERT INTO GeneratedResponses
            (experiment_id, input_text, response_text, timestamp)
            VALUES (?, ?, ?, ?)
        """, (experiment_id, input_text, response_text, datetime.now().isoformat()))
        self.conn.commit()

    def close(self):
        self.conn.close()

In [15]:
# Cell 11 — Main pipeline (launch with torchrun for true multi-GPU)
import torch

print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

df = pd.read_csv("/kaggle/input/bengali-empathetic-conversations-corpus/BengaliEmpatheticConversationsCorpus .csv")
processor    = DatasetProcessor(MODEL_NAME)
dataset      = processor.preprocess(df)
dataset_dict = split_dataset(dataset, test_size=0.1)

print(f"Train: {len(dataset_dict['train'])}  |  Eval: {len(dataset_dict['eval'])}")

strategy   = UnslothStrategy(MODEL_NAME)
fine_tuner = LLAMAFineTuner(strategy)
model, tokenizer = fine_tuner.run(dataset_dict)

FastLanguageModel.for_inference(model)

evaluator  = Evaluator(model, tokenizer)
perplexity = evaluator.compute_perplexity(dataset_dict["eval"])
print(f"\nPerplexity: {perplexity:.4f}")

metrics, predictions, references = evaluator.evaluate_on_samples(dataset_dict["eval"], n_samples=50)
print("\nMetrics:", metrics)

logger = ExperimentLogger()
lora_config = {
    "r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
    "target_modules": ["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"]
}
exp_id = logger.log_experiment(
    model_name  = MODEL_NAME,
    lora_config = lora_config,
    train_loss  = strategy.train_loss,
    val_loss    = strategy.val_loss,
    metrics     = {**metrics, "perplexity": perplexity}
)
for inp, resp in zip(references[:10], predictions[:10]):
    logger.log_response(exp_id, inp, resp)

logger.close()
print(f"\nExperiment #{exp_id} logged.")

GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Train: 4500  |  Eval: 500
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,500 | Num Epochs = 2 | Total steps = 564
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,0.950400
20,0.728900
30,0.619800
40,0.586800
50,0.561500
60,0.538300
70,0.527000
80,0.503600
90,0.508300
100,0.518100


Eval Loss: 0.4051



Perplexity: 1.5030
=== FIRST SAMPLE (first 400 chars) ===
'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\nআপনি একজন সহানুভূতিশীল বাংলা সহায়তাকারী। সবসময় বাংলায় উত্তর দিন।<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nবিষয়: সৎ কন্যা আজ ওভারডোজ করেছে\n\nশিরোনাম: সৎ কন্যা এবং একটি সৎপুত্র আমি আমার সৎ কন্যাকে খুঁজে পেয়েছি\n\nপ্রশ্ন:  আমার তিনটি সন্তান আছে: একটি'

Evaluated 50 samples
--- Example prediction ---
হ্যালো, এন্ড অ্যাপগোলি!  আর.কে. ফ্রান্সিস   কঠিন পরিস্থিতিতে আরামদাযঃ  স্নাযড়ণ, উদ্বেগ, স্তব্ধতা এরকম আরো বহুল প্রেক্ষাপটে অনুসরণ করা স্ব-যত্নের দ্বারা   আপত্বমূলক জীবনধারা, দুর্ঘটনা এ বা এ, কম্পাউন্
--- Example reference ---
অন্যকে দোষারোপ করা আমাদের সবচেয়ে বড় মোকাবিলার ব্যবস্থাগুলির মধ্যে একটি।  এটি শুধু আসক্তির জন্য নয়, এবং প্রায়শই আমরা আমাদের কাছের লোকদের দোষ দেই।  আপনি তাদের আপনাকে দোষারোপ করা থেকে থামাতে পারবেন ন

Metrics: {'BLEU': 0.0376, 'ROUGE1': np.float64(0.

In [16]:
# Cell 11b — Save & Reload Fine-Tuned Model

SAVE_PATH = "/kaggle/working/bengali-empathetic-llama"   # persisted in Kaggle output

# ── SAVING (run once after training) ──────────────────────────────────────────
# Saves only the LoRA adapter weights (~80 MB), not the full 8B model
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ LoRA adapter + tokenizer saved to: {SAVE_PATH}")


# ── RELOADING (run in a fresh session instead of re-training) ─────────────────
# Uncomment the block below and run it at the top of a new session.
# Make sure Cell 0 (pip installs) and Cell 3 (HF login) have been run first.

"""
from unsloth import FastLanguageModel

SAVE_PATH  = "/kaggle/working/bengali-empathetic-llama"
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = SAVE_PATH,      # ← load adapter from disk
    max_seq_length = 2048,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

FastLanguageModel.for_inference(model)  # enable fast inference mode
print(" Model reloaded from saved adapter — no retraining needed.")
"""

✅ LoRA adapter + tokenizer saved to: /kaggle/working/bengali-empathetic-llama


'\nfrom unsloth import FastLanguageModel\n\nSAVE_PATH  = "/kaggle/working/bengali-empathetic-llama"\nMODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"\n\nmodel, tokenizer = FastLanguageModel.from_pretrained(\n    model_name     = SAVE_PATH,      # ← load adapter from disk\n    max_seq_length = 2048,\n    load_in_4bit   = True,\n    token          = HF_TOKEN,\n)\n\nFastLanguageModel.for_inference(model)  # enable fast inference mode\nprint(" Model reloaded from saved adapter — no retraining needed.")\n'

In [17]:
# Cell 12 — Sample responses on test prompts (FIXED)

test_cases = [
    {"topic": "দুঃখ",      "title": "মন খারাপ",       "question": "আমার আজকে খুব মন খারাপ, কী করব?"},
    {"topic": "একাকীত্ব",  "title": "একা অনুভব করছি", "question": "সবাই আমাকে ছেড়ে চলে গেছে মনে হচ্ছে।"},
    {"topic": "উদ্বেগ",    "title": "পরীক্ষার ভয়",   "question": "পরীক্ষার আগে অনেক ভয় লাগছে, কীভাবে শান্ত হব?"},
]

def build_prompt(topic, title, question):
    messages = [
        {
            "role": "system",
            "content": "আপনি একজন সহানুভূতিশীল বাংলা সহায়তাকারী। সবসময় বাংলায় উত্তর দিন।"
        },
        {
            "role": "user",
            "content": f"বিষয়: {topic}\n\nশিরোনাম: {title}\n\nপ্রশ্ন: {question}"
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

def generate_clean(prompt: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3800).to("cuda")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|eot_id|>")
        )
    new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# ── Reopen logger — was closed at end of Cell 11 ──
logger = ExperimentLogger()

print("=" * 60)
print("SAMPLE GENERATED RESPONSES (Fixed)")
print("=" * 60)

for i, tc in enumerate(test_cases, 1):
    prompt   = build_prompt(tc["topic"], tc["title"], tc["question"])
    response = generate_clean(prompt)
    print(f"\n[Prompt {i}]")
    print(f"বিষয়: {tc['topic']} | শিরোনাম: {tc['title']}")
    print(f"প্রশ্ন: {tc['question']}")
    print(f"\n[Response {i}]")
    print(response)
    print("-" * 60)
    logger.log_response(exp_id, tc["question"], response)

# ── Close ONCE after loop — Cell 14 will reopen its own logger ──
logger.close()

SAMPLE GENERATED RESPONSES (Fixed)

[Prompt 1]
বিষয়: দুঃখ | শিরোনাম: মন খারাপ
প্রশ্ন: আমার আজকে খুব মন খারাপ, কী করব?

[Response 1]
চাইলে গতকাল যা ঘটেছে সে সম্পর্কে লেখুন। অনেক মানোহর জিনিস হযডাউন এবং ফ্লার্ট. আপত্রবাহী হওযমি!
------------------------------------------------------------

[Prompt 2]
বিষয়: একাকীত্ব | শিরোনাম: একা অনুভব করছি
প্রশ্ন: সবাই আমাকে ছেড়ে চলে গেছে মনে হচ্ছে।

[Response 2]
আরে, এটা ঠিক আছে. খুব খারাপ অনেক লোক হতে পারে না। আপাতত স্থানীযবধ করুন এবং সম্প্রেক্ষিত পরিস্থিতি সংশোধন করার চেষ্টা করা হবে বা আশা কিছু ভালো হওযক করে আঙুল উঠবে
------------------------------------------------------------

[Prompt 3]
বিষয়: উদ্বেগ | শিরোনাম: পরীক্ষার ভয়
প্রশ্ন: পরীক্ষার আগে অনেক ভয় লাগছে, কীভাবে শান্ত হব?

[Response 3]
এটি খুবই সাধারণ এবং আপত্রুপ! আমি চাই যে আলোচনা আকর্ষণীযঈ습니다. এখন, আসুন এটা থেকে আরও ভাল হওযয করা যাক.
------------------------------------------------------------


In [18]:
# Cell 13 — Metrics summary table
import pandas as pd

results_df = pd.DataFrame([{
    "Model":       MODEL_NAME,
    "Perplexity":  round(perplexity, 4),
    "BLEU":        metrics["BLEU"],
    "ROUGE-1":     metrics["ROUGE1"],
    "ROUGE-2":     metrics["ROUGE2"],
    "ROUGE-L":     metrics["ROUGEL"],
    "Train Loss":  round(strategy.train_loss, 4),
    "Val Loss":    round(strategy.val_loss, 4) if strategy.val_loss else "N/A",
}])

print(results_df.to_string(index=False))

                                Model  Perplexity   BLEU  ROUGE-1  ROUGE-2  ROUGE-L  Train Loss  Val Loss
meta-llama/Meta-Llama-3.1-8B-Instruct       1.503 0.0376   0.4468   0.1908   0.2888      0.4955    0.4051


In [19]:
# Cell 14 — Human Evaluation Rubric (Kaggle-compatible)
# After running, read the responses printed above and fill in your scores below

HUMAN_EVAL_CRITERIA = {
    "Empathy":    "Does the response acknowledge and validate the user's emotions?",
    "Relevance":  "Is the response relevant to the topic and question asked?",
    "Fluency":    "Is the Bengali text fluent and grammatically correct?",
    "Coherence":  "Is the response logically coherent and meaningful?",
    "Helpfulness":"Does the response offer useful support or guidance?",
}

test_cases = [
    {"topic": "দুঃখ",      "title": "মন খারাপ",       "question": "আমার আজকে খুব মন খারাপ, কী করব?"},
    {"topic": "একাকীত্ব",  "title": "একা অনুভব করছি", "question": "সবাই আমাকে ছেড়ে চলে গেছে মনে হচ্ছে।"},
    {"topic": "উদ্বেগ",    "title": "পরীক্ষার ভয়",   "question": "পরীক্ষার আগে অনেক ভয় লাগছে, কীভাবে শান্ত হব?"},
]

# ── STEP 1: Generate and print all responses first ──────────────────────────
print("=" * 70)
print("GENERATED RESPONSES FOR HUMAN EVALUATION")
print("=" * 70)

responses = []
for i, tc in enumerate(test_cases, 1):
    prompt   = build_prompt(tc["topic"], tc["title"], tc["question"])
    response = generate_clean(prompt)
    responses.append(response)
    print(f"\n[Sample {i}] বিষয়: {tc['topic']} | শিরোনাম: {tc['title']}")
    print(f"প্রশ্ন: {tc['question']}")
    print(f"মডেলের উত্তর:\n{response}")
    print("─" * 70)

# ── STEP 2: Fill in your scores here after reading the responses above ───────
# Scale: 1 = poor, 5 = excellent
MANUAL_SCORES = [
    # Sample 1
    {"Empathy": 2, "Relevance": 2, "Fluency": 1, "Coherence": 1, "Helpfulness": 1},
    # Sample 2
    {"Empathy": 2, "Relevance": 2, "Fluency": 1, "Coherence": 1, "Helpfulness": 1},
    # Sample 3
    {"Empathy": 2, "Relevance": 2, "Fluency": 1, "Coherence": 1, "Helpfulness": 1},
]

# ── STEP 3: Build summary table ──────────────────────────────────────────────
human_eval_records = []
logger2 = ExperimentLogger()

for i, (tc, response, scores) in enumerate(zip(test_cases, responses, MANUAL_SCORES), 1):
    avg = round(sum(scores.values()) / len(scores), 2)
    human_eval_records.append({
        "Sample":  i,
        "Topic":   tc["topic"],
        "Question": tc["question"],
        **scores,
        "Average": avg,
    })
    logger2.log_response(exp_id, tc["question"], response)

logger2.close()

human_df = pd.DataFrame(human_eval_records)
display_cols = ["Sample", "Topic"] + list(HUMAN_EVAL_CRITERIA.keys()) + ["Average"]

print(f"\n{'='*70}")
print("HUMAN EVALUATION SUMMARY")
print("=" * 70)
print(human_df[display_cols].to_string(index=False))

print(f"\n{'─'*70}")
print("Overall Average per Criterion:")
for criterion in list(HUMAN_EVAL_CRITERIA.keys()) + ["Average"]:
    print(f"  {criterion}: {human_df[criterion].mean():.2f} / 5")

GENERATED RESPONSES FOR HUMAN EVALUATION

[Sample 1] বিষয়: দুঃখ | শিরোনাম: মন খারাপ
প্রশ্ন: আমার আজকে খুব মন খারাপ, কী করব?
মডেলের উত্তর:
হ্যাঁ, কিছুটা আরাম করা সাধারণত সফল. আপত্থি এর চেযড ইঙ্গিত দিয়ে বেশি ঘন্টার জন্য অনুপযুক্ত. এটি এনার্জি ব্যবহার করে এবং আসলে কঠোরভাবে কাজ করার পরে আলাদা হওযঈ습니다।
──────────────────────────────────────────────────────────────────────

[Sample 2] বিষয়: একাকীত্ব | শিরোনাম: একা অনুভব করছি
প্রশ্ন: সবাই আমাকে ছেড়ে চলে গেছে মনে হচ্ছে।
মডেলের উত্তর:
আর খুশি হওযডা, আপত্ট! এটি কোন মানৈক্যের জন্য ঘটে থাকে. আত্ম-প্যার এবং অন্যদের সঙ্গে থেকো যোগাযো...
──────────────────────────────────────────────────────────────────────

[Sample 3] বিষয়: উদ্বেগ | শিরোনাম: পরীক্ষার ভয়
প্রশ্ন: পরীক্ষার আগে অনেক ভয় লাগছে, কীভাবে শান্ত হব?
মডেলের উত্তর:
পরৈক্ক্ষা খুব কঠিন. আপত্রুণ থাকা স্বাভাবিক কিন্তু কিছু চটকদার ধারণাও আছে যা আমি সত্যিই সাহায্য করেছি। যেমন প্রেশার মানিয়ে নেওযআমরা করি না তা ঘটানোর জন্য গবেষণার ফলাফল এবং তারপরে এই উপাযলগুলির এখানে এর এরকম আক
─────────────────